# Exp15 — Cross-language agreement dissociation (Gemma-2 2B/9B/27B)

**Purpose.** Test whether a high-ε task (subject–verb number agreement) dissociates
with scale while a low-ε task (IOI, from Exp13/14) stays shared. The ε-framework's
prediction: EN→FR head-output patching compat for agreement drops below 1.0 and
drops more at 27B than at 2B; the (IOI − agreement) gap widens with scale. If
compat stays ~1.0 for agreement too (same as IOI), the ε framework is in trouble
and the Gemma data would be consistent with Lim-et-al-style uniform dissociation
being wrong (or operating only at the representational, not computational, level).

**Pipeline.** Same shape as Exp13/14: rank heads by their contribution to the
agreement logit-diff per language per model; ablate the top-K; then head-output
patching EN→FR with a same-language different-template floor control. All cells
below are self-contained and reuse the local Gemma paths and GPU_ID convention
from Exp14.

**Reads** `exp14_results/<model>.json` if present, to produce the IOI vs agreement
side-by-side at the end. Not required to run.


## Cell 1 — Setup

In [1]:
import os
os.environ["http_proxy"]  = "http://proxy61.iitd.ac.in:3128"
os.environ["https_proxy"] = "http://proxy61.iitd.ac.in:3128"
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "<REDACTED>")
HF_TOKEN = os.environ.get("HF_TOKEN")
os.environ["HF_HUB_OFFLINE"] = "1"


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"  # GPU_ID sentinel, same as Exp13/14

import json, random, gc
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)

MODEL_PATHS = {
    "2b_base": "google/gemma-2-2b",              # Hub id, not a local folder
    "9b_base": "/home/models/gemma-2-9b",
    "9b_it":   "/home/models/gemma-2-9b-it",
    "27b_it":  "/home/models/gemma-2-27b-it",
}

OUT_DIR = Path("exp15_results"); OUT_DIR.mkdir(exist_ok=True)
EXP14_DIR = Path("exp14_results")  # for IOI compat side-by-side (optional)

SEEDS = [0, 7, 13]  # 3-seed convention from Exp13/14
N_PROMPTS = 120     # per condition per seed
TOPK_HEADS = 10     # match Exp14's K=10 head-output patching cell

print("CUDA:", torch.cuda.is_available(), "| device:", DEVICE)


/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA: True | device: cuda


## Cell 2 — Agreement task

Number-agreement with attractor interference: `[Det N1_sg Prep (Det) N2_pl] → V`.
The correct next-token verb is singular (agreeing with N1); the plural attractor
N2 is the standard interference that forces real agreement tracking rather than
local token statistics. Verb pairs chosen for single-token encoding under Gemma's
sentencepiece (verified in Cell 4).

Two FR conditions parallel to Exp14's `YO` vs `YOs` split:
- **`fr_nat`**: FR templates + FR nouns (natural)
- **`fr_matched`**: FR templates + EN nouns, isolating structural/grammatical
  agreement from lexical variation. This is the condition used for cross-language
  patching, same role as Exp14's English-names-in-non-EN templates.


In [3]:
# EN templates — 6 structural variants
EN_TEMPLATES = [
    "The {N1} near the {N2}",
    "The {N1} behind the {N2}",
    "The {N1} next to the {N2}",
    "The {N1} by the {N2}",
    "The {N1} with the {N2}",
    "The {N1} among the {N2}",
]

# FR templates — matched structure; masculine singular to sidestep gender
FR_TEMPLATES = [
    "Le {N1} près des {N2}",
    "Le {N1} derrière les {N2}",
    "Le {N1} à côté des {N2}",
    "Le {N1} par les {N2}",
    "Le {N1} avec les {N2}",
    "Le {N1} parmi les {N2}",
]

# 20 (sg, pl) EN pairs, concrete countable nouns
EN_NOUN_PAIRS = [
    ("cat","cats"),("dog","dogs"),("book","books"),("key","keys"),
    ("hat","hats"),("bird","birds"),("horse","horses"),("phone","phones"),
    ("doctor","doctors"),("teacher","teachers"),("student","students"),("friend","friends"),
    ("brother","brothers"),("boy","boys"),("father","fathers"),("player","players"),
    ("author","authors"),("painter","painters"),("singer","singers"),("writer","writers"),
]

# 20 FR masculine (sg, pl) pairs, chosen so Le/les article stays consistent
FR_NOUN_PAIRS = [
    ("chat","chats"),("chien","chiens"),("livre","livres"),("stylo","stylos"),
    ("chapeau","chapeaux"),("oiseau","oiseaux"),("cheval","chevaux"),("téléphone","téléphones"),
    ("docteur","docteurs"),("professeur","professeurs"),("étudiant","étudiants"),("ami","amis"),
    ("frère","frères"),("garçon","garçons"),("père","pères"),("joueur","joueurs"),
    ("auteur","auteurs"),("peintre","peintres"),("chanteur","chanteurs"),("écrivain","écrivains"),
]

# Verb pairs: (sg_form, pl_form). sg is the CORRECT continuation under our sg-subject design.
EN_VERB_PAIRS = [(" is", " are"), (" was", " were"), (" has", " have")]
FR_VERB_PAIRS = [(" est", " sont"), (" était", " étaient"), (" a", " ont")]


## Cell 3 — Prompt generation

In [4]:
def build_prompts(templates, noun_pairs, verb_pairs, n, seed):
    rng = random.Random(seed)
    out = []
    for _ in range(n):
        tpl = rng.choice(templates)
        n1_sg, _ = rng.choice(noun_pairs)
        others = [p for p in noun_pairs if p[0] != n1_sg]
        _, n2_pl = rng.choice(others)
        v_sg, v_pl = rng.choice(verb_pairs)
        text = tpl.format(N1=n1_sg, N2=n2_pl)
        out.append({"text": text, "correct": v_sg, "incorrect": v_pl,
                    "N1": n1_sg, "N2": n2_pl, "template": tpl})
    return out


def build_all_conditions(seed=0, n=N_PROMPTS):
    # EN natural / FR natural / FR with EN nouns (matched for cross-lang patching)
    return {
        "en_nat":     build_prompts(EN_TEMPLATES, EN_NOUN_PAIRS, EN_VERB_PAIRS, n, seed),
        "fr_nat":     build_prompts(FR_TEMPLATES, FR_NOUN_PAIRS, FR_VERB_PAIRS, n, seed),
        "fr_matched": build_prompts(FR_TEMPLATES, EN_NOUN_PAIRS, FR_VERB_PAIRS, n, seed),
    }


# quick peek
_demo = build_all_conditions(seed=0, n=3)
for k, v in _demo.items():
    print(k)
    for p in v:
        print(f"  '{p['text']}' | correct={p['correct']!r} incorrect={p['incorrect']!r}")


en_nat
  'The boy by the dogs' | correct=' was' incorrect=' were'
  'The player with the brothers' | correct=' was' incorrect=' were'
  'The friend by the writers' | correct=' is' incorrect=' are'
fr_nat
  'Le garçon par les chiens' | correct=' était' incorrect=' étaient'
  'Le joueur avec les frères' | correct=' était' incorrect=' étaient'
  'Le ami par les écrivains' | correct=' est' incorrect=' sont'
fr_matched
  'Le boy par les dogs' | correct=' était' incorrect=' étaient'
  'Le player avec les brothers' | correct=' était' incorrect=' étaient'
  'Le friend par les writers' | correct=' est' incorrect=' sont'


## Cell 4 — Model loader + single-token verification

Loads a Gemma-2 checkpoint in bf16 on the pinned GPU, verifies that every verb
pair encodes to a single token under this model's tokenizer, and returns the
model/tokenizer and head/layer config needed by the hooks below. If any verb
pair fails single-token check, that pair is dropped for this model — logit-diff
is only well-defined on the first mismatching token, so multi-token pairs would
silently measure the wrong thing.


In [5]:
def load_model(model_path):
    tok = AutoTokenizer.from_pretrained(model_path)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16, device_map=DEVICE
    )
    mdl.eval()
    cfg = {
        "n_layers": mdl.config.num_hidden_layers,
        "n_heads":  mdl.config.num_attention_heads,
        "head_dim": mdl.config.head_dim,
    }
    return mdl, tok, cfg


def verify_verb_pairs(tok, pairs):
    good = []
    for (a, b) in pairs:
        ta = tok.encode(a, add_special_tokens=False)
        tb = tok.encode(b, add_special_tokens=False)
        if len(ta) == 1 and len(tb) == 1:
            good.append((a, b))
        else:
            print(f"  dropping {a!r}/{b!r}: encodes to {ta}/{tb}")
    return good


def filter_prompts_by_verbs(prompts, kept_pairs):
    kept = {v for pair in kept_pairs for v in pair}
    return [p for p in prompts if p["correct"] in kept and p["incorrect"] in kept]


def unload(mdl):
    del mdl
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## Cell 5 — Logit-diff eval

For each prompt, run the model once and read the final-position logit of
`correct_verb[0]` minus `incorrect_verb[0]`. Returns per-prompt diffs so we can
compute both mean logit-diff and 0/1 accuracy (`diff > 0`). This is the primary
metric throughout.


In [6]:
def logit_diffs(mdl, tok, prompts):
    diffs = np.zeros(len(prompts), dtype=np.float32)
    for i, p in enumerate(prompts):
        enc = tok(p["text"], return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = mdl(**enc)
        logits = out.logits[0, -1]
        c_tok = tok.encode(p["correct"],   add_special_tokens=False)[0]
        i_tok = tok.encode(p["incorrect"], add_special_tokens=False)[0]
        diffs[i] = float(logits[c_tok] - logits[i_tok])
    return diffs


## Cell 6 — Section A — Baselines

For each scale, baseline logit-diff on `en_nat`, `fr_nat`, `fr_matched`. Any
condition whose accuracy sits near chance (0.5) at a given scale means the model
doesn't have the circuit we'd try to study — the ablation/patching numbers that
follow would be meaningless. Rule of thumb before proceeding: accuracy ≥ 0.75 on
each used condition at each scale, matching the Exp14 preflight for IOI.


In [7]:
baseline_results = {}

for mk, mp in MODEL_PATHS.items():
    print(f"\n=== {mk} ===")
    mdl, tok, cfg = load_model(mp)
    print(f"  layers={cfg['n_layers']} heads={cfg['n_heads']} head_dim={cfg['head_dim']}")

    en_verbs_ok = verify_verb_pairs(tok, EN_VERB_PAIRS)
    fr_verbs_ok = verify_verb_pairs(tok, FR_VERB_PAIRS)
    print(f"  EN verb pairs kept: {len(en_verbs_ok)}/{len(EN_VERB_PAIRS)}")
    print(f"  FR verb pairs kept: {len(fr_verbs_ok)}/{len(FR_VERB_PAIRS)}")

    cond = build_all_conditions(seed=0)
    cond["en_nat"]     = filter_prompts_by_verbs(cond["en_nat"],     en_verbs_ok)
    cond["fr_nat"]     = filter_prompts_by_verbs(cond["fr_nat"],     fr_verbs_ok)
    cond["fr_matched"] = filter_prompts_by_verbs(cond["fr_matched"], fr_verbs_ok)

    row = {}
    for name, prompts in cond.items():
        d = logit_diffs(mdl, tok, prompts)
        row[name] = {"n": len(prompts),
                     "mean_diff": float(d.mean()),
                     "acc":       float((d > 0).mean())}
        print(f"  {name:11s}: n={row[name]['n']:3d}  mean_diff={row[name]['mean_diff']:+.3f}  acc={row[name]['acc']:.3f}")

    baseline_results[mk] = {
        "cfg": cfg,
        "en_verbs_kept": en_verbs_ok,
        "fr_verbs_kept": fr_verbs_ok,
        "conditions": row,
    }
    with open(OUT_DIR / f"{mk}_baseline.json", "w") as f:
        json.dump(baseline_results[mk], f, indent=2, default=str)

    unload(mdl)

print("\nbaseline done.")



=== 2b_base ===


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 288/288 [00:00<00:00, 826.92it/s]


  layers=26 heads=8 head_dim=256
  EN verb pairs kept: 3/3
  FR verb pairs kept: 3/3
  en_nat     : n=120  mean_diff=+2.763  acc=0.967
  fr_nat     : n=120  mean_diff=+4.134  acc=0.992
  fr_matched : n=120  mean_diff=+2.139  acc=0.950

=== 9b_base ===


Loading weights: 100%|██████████| 464/464 [00:01<00:00, 400.67it/s]


  layers=42 heads=16 head_dim=256
  EN verb pairs kept: 3/3
  FR verb pairs kept: 3/3
  en_nat     : n=120  mean_diff=+3.207  acc=0.983
  fr_nat     : n=120  mean_diff=+3.588  acc=0.917
  fr_matched : n=120  mean_diff=+1.723  acc=0.975

=== 9b_it ===


Loading weights: 100%|██████████| 464/464 [00:02<00:00, 225.95it/s]


  layers=42 heads=16 head_dim=256
  EN verb pairs kept: 3/3
  FR verb pairs kept: 3/3
  en_nat     : n=120  mean_diff=+3.700  acc=0.958
  fr_nat     : n=120  mean_diff=+3.414  acc=0.975
  fr_matched : n=120  mean_diff=+1.654  acc=0.942

=== 27b_it ===


Loading weights: 100%|██████████| 508/508 [00:05<00:00, 88.28it/s]


  layers=46 heads=32 head_dim=128
  EN verb pairs kept: 3/3
  FR verb pairs kept: 3/3
  en_nat     : n=120  mean_diff=+3.396  acc=0.983
  fr_nat     : n=120  mean_diff=+2.857  acc=0.883
  fr_matched : n=120  mean_diff=+1.118  acc=0.800

baseline done.


## Cell 7 — Attention-head intercept (ablate / capture / patch)

Standard pre-hook on `self_attn.o_proj`: its input is the concatenated
head outputs with shape `[batch, seq, n_heads * head_dim]`. Reshape by heads,
zero or replace the specified head's slice, reshape back. Returns a context
manager that cleans hooks up on exit so stacked experiments don't leak state.

Three modes:
- `"ablate"`: zero the specified heads at **all** positions.
- `"capture"`: save the head output tensor at the **final** position for later patching.
- `"patch"`: write the captured tensor into the specified heads at the final position.

The pattern matches Exp14's head-output patching: intervene on the heads' output
to `o_proj`, not on `o_proj`'s result (which would mix heads back together).


In [8]:
from contextlib import contextmanager


@contextmanager
def head_hooks(mdl, cfg, mode, heads, patch_payload=None, capture_store=None):
    """
    heads: iterable of (layer_idx, head_idx)
    mode in {"ablate", "capture", "patch"}
    patch_payload: dict {(layer_idx, head_idx): tensor [head_dim]} for "patch"
    capture_store: dict to fill for "capture"; keyed by (layer_idx, head_idx)
    """
    handles = []
    heads_by_layer = defaultdict(list)
    for (L, H) in heads:
        heads_by_layer[L].append(H)

    head_dim = cfg["head_dim"]

    def make_hook(L, H_list):
        def pre_hook(module, inputs):
            (x,) = inputs  # [B, S, n_heads * head_dim]
            B, S, HD = x.shape
            x = x.view(B, S, HD // head_dim, head_dim)
            if mode == "ablate":
                for H in H_list:
                    x[:, :, H, :] = 0.0
            elif mode == "capture":
                for H in H_list:
                    capture_store[(L, H)] = x[:, -1, H, :].detach().clone()
            elif mode == "patch":
                for H in H_list:
                    key = (L, H)
                    if key in patch_payload:
                        v = patch_payload[key].to(x.device, dtype=x.dtype)
                        # v: [B, head_dim] or [head_dim]; broadcast onto final pos
                        if v.dim() == 1:
                            x[:, -1, H, :] = v
                        else:
                            x[:, -1, H, :] = v
            return (x.view(B, S, HD),)
        return pre_hook

    for L, H_list in heads_by_layer.items():
        layer = mdl.model.layers[L].self_attn
        h = layer.o_proj.register_forward_pre_hook(make_hook(L, H_list))
        handles.append(h)

    try:
        yield
    finally:
        for h in handles:
            h.remove()


def logit_diffs_with_hooks(mdl, tok, prompts, mode, heads,
                           patch_payload=None, capture_store=None):
    diffs = np.zeros(len(prompts), dtype=np.float32)
    for i, p in enumerate(prompts):
        enc = tok(p["text"], return_tensors="pt").to(DEVICE)
        # per-prompt capture / patch so payloads are prompt-local (not pooled)
        if mode == "patch":
            # patch_payload is a list aligned to prompts
            per_prompt_payload = patch_payload[i]
            with head_hooks(mdl, cfg_for_model(mdl), mode="patch",
                            heads=list(per_prompt_payload.keys()),
                            patch_payload=per_prompt_payload):
                out = mdl(**enc)
        elif mode == "capture":
            # capture_store expected as list; fill entry i
            store = {}
            with head_hooks(mdl, cfg_for_model(mdl), mode="capture",
                            heads=heads, capture_store=store):
                out = mdl(**enc)
            capture_store[i] = store
        elif mode == "ablate":
            with head_hooks(mdl, cfg_for_model(mdl), mode="ablate", heads=heads):
                out = mdl(**enc)
        else:
            raise ValueError(mode)

        logits = out.logits[0, -1]
        c_tok = tok.encode(p["correct"],   add_special_tokens=False)[0]
        i_tok = tok.encode(p["incorrect"], add_special_tokens=False)[0]
        diffs[i] = float(logits[c_tok] - logits[i_tok])
    return diffs


def cfg_for_model(mdl):
    return {"n_layers": mdl.config.num_hidden_layers,
            "n_heads":  mdl.config.num_attention_heads,
            "head_dim": mdl.config.head_dim}


## Cell 8 — Section B — Head ranking per language per model

For each (layer, head), ablate it alone across the prompt set in that language;
rank heads by `clean_mean − ablated_mean` (bigger = more load-bearing for
agreement). This is the per-model, per-language head ranking equivalent to
Exp13's IOI ranking. Expensive — one forward pass per (prompt × head). We cache
to `exp15_results/<model>_rank_<lang>.json` and skip re-running if present.


In [9]:
def rank_heads_for_lang(mdl, tok, prompts, cfg, max_heads=None):
    n_layers, n_heads = cfg["n_layers"], cfg["n_heads"]
    clean = logit_diffs(mdl, tok, prompts).mean()

    scores = np.zeros((n_layers, n_heads), dtype=np.float32)
    for L in range(n_layers):
        for H in range(n_heads):
            d = logit_diffs_with_hooks(mdl, tok, prompts, mode="ablate",
                                       heads=[(L, H)])
            scores[L, H] = clean - d.mean()  # positive = ablating hurt
    ranked = sorted(
        [(int(L), int(H), float(scores[L, H])) for L in range(n_layers) for H in range(n_heads)],
        key=lambda x: -x[2]
    )
    return {"clean_mean": float(clean), "scores": scores.tolist(), "ranked": ranked}


def rank_or_load(mk, lang_key, mdl, tok, prompts, cfg):
    cache = OUT_DIR / f"{mk}_rank_{lang_key}.json"
    if cache.exists():
        print(f"  [cache] {cache.name}")
        with open(cache) as f: return json.load(f)
    print(f"  ranking heads for {mk}/{lang_key} — {cfg['n_layers']}x{cfg['n_heads']} = {cfg['n_layers']*cfg['n_heads']} ablations")
    r = rank_heads_for_lang(mdl, tok, prompts, cfg)
    with open(cache, "w") as f: json.dump(r, f)
    return r


# To actually run ranking, loop per model.
# ~Expect several minutes per model on H100. Uncomment to execute:
#
# for mk, mp in MODEL_PATHS.items():
#     mdl, tok, cfg = load_model(mp)
#     en_verbs = verify_verb_pairs(tok, EN_VERB_PAIRS)
#     fr_verbs = verify_verb_pairs(tok, FR_VERB_PAIRS)
#     cond = build_all_conditions(seed=0)
#     cond["en_nat"]     = filter_prompts_by_verbs(cond["en_nat"],     en_verbs)
#     cond["fr_matched"] = filter_prompts_by_verbs(cond["fr_matched"], fr_verbs)
#     rank_or_load(mk, "en_nat",     mdl, tok, cond["en_nat"],     cfg)
#     rank_or_load(mk, "fr_matched", mdl, tok, cond["fr_matched"], cfg)
#     unload(mdl)


## Cell 9 — Section C — Head ablation (Exp13-equivalent for agreement)

Table analogous to Exp13 §2a:

| scale | EN-ranked top-K → EN | → FR-matched | FR-ranked top-K → FR-matched | → EN | random K heads |
|---|---|---|---|---|---|

Retained logit-diff fraction = `(ablated_mean) / (clean_mean)`. Shared circuit
→ EN-ranked heads hurt both EN and FR. Duplicate per-language circuits →
EN-ranked heads barely touch FR. Compare the drop pattern directly to Exp13's
IOI numbers.


In [10]:
def retained_fraction(clean_mean, ablated_mean):
    if abs(clean_mean) < 1e-6:
        return float("nan")
    return float(ablated_mean / clean_mean)


def ablation_study(mk, mp, topk=TOPK_HEADS, n_random=TOPK_HEADS, seed=0):
    mdl, tok, cfg = load_model(mp)
    en_verbs = verify_verb_pairs(tok, EN_VERB_PAIRS)
    fr_verbs = verify_verb_pairs(tok, FR_VERB_PAIRS)
    cond = build_all_conditions(seed=seed)
    en_prompts = filter_prompts_by_verbs(cond["en_nat"],     en_verbs)
    fr_prompts = filter_prompts_by_verbs(cond["fr_matched"], fr_verbs)

    en_rank = rank_or_load(mk, "en_nat",     mdl, tok, en_prompts, cfg)
    fr_rank = rank_or_load(mk, "fr_matched", mdl, tok, fr_prompts, cfg)

    en_heads = [(L, H) for (L, H, _s) in en_rank["ranked"][:topk]]
    fr_heads = [(L, H) for (L, H, _s) in fr_rank["ranked"][:topk]]

    rng = random.Random(seed)
    all_heads = [(L, H) for L in range(cfg["n_layers"]) for H in range(cfg["n_heads"])]
    random_heads = rng.sample(all_heads, n_random)

    clean_en = logit_diffs(mdl, tok, en_prompts).mean()
    clean_fr = logit_diffs(mdl, tok, fr_prompts).mean()

    def ablate_and_measure(heads, prompts):
        d = logit_diffs_with_hooks(mdl, tok, prompts, mode="ablate", heads=heads)
        return float(d.mean())

    rows = {
        "clean_en": float(clean_en),
        "clean_fr": float(clean_fr),
        "en_heads_to_en": ablate_and_measure(en_heads,     en_prompts),
        "en_heads_to_fr": ablate_and_measure(en_heads,     fr_prompts),
        "fr_heads_to_fr": ablate_and_measure(fr_heads,     fr_prompts),
        "fr_heads_to_en": ablate_and_measure(fr_heads,     en_prompts),
        "rand_to_en":     ablate_and_measure(random_heads, en_prompts),
        "rand_to_fr":     ablate_and_measure(random_heads, fr_prompts),
        "topk": topk,
    }

    rows["retained"] = {
        "en_heads→en":   retained_fraction(clean_en, rows["en_heads_to_en"]),
        "en_heads→fr":   retained_fraction(clean_fr, rows["en_heads_to_fr"]),
        "fr_heads→fr":   retained_fraction(clean_fr, rows["fr_heads_to_fr"]),
        "fr_heads→en":   retained_fraction(clean_en, rows["fr_heads_to_en"]),
        "random→en":     retained_fraction(clean_en, rows["rand_to_en"]),
        "random→fr":     retained_fraction(clean_fr, rows["rand_to_fr"]),
    }

    with open(OUT_DIR / f"{mk}_ablation.json", "w") as f:
        json.dump(rows, f, indent=2)

    print(f"[{mk}] retained fractions:")
    for k, v in rows["retained"].items():
        print(f"  {k:14s} = {v:+.3f}")

    unload(mdl)
    return rows


# To run: ablation_study("2b_base", MODEL_PATHS["2b_base"]) etc.
# All four:
# for mk, mp in MODEL_PATHS.items():
#     ablation_study(mk, mp)


## Cell 10 — Section D — Head-output patching (Exp14-equivalent for agreement)

Compat metric from Exp14: `compat = (patched − ablated) / (clean − ablated)`,
with 1.0 = EN heads' output is a drop-in replacement for FR's.

**Required floor control** (same shape as Exp14's same-language-different-template):
patch EN heads **within English** across different templates. If floor ~1.0, the
patching methodology itself is sound and cross-language numbers can be trusted.
If floor is far from 1.0, something is broken before interpreting cross-language
numbers.

Pairing: each FR prompt gets paired with an EN "donor" prompt (same template
index, same noun positions, grammatically matched). Head outputs captured at
the donor's final position (sg subject, pl attractor), then written at the FR
target's final position. The hook writes per-prompt, so there's no activation
pooling.


In [11]:
def pair_prompts(en_prompts, fr_prompts):
    """Simple index-aligned pairing. Both lists built with the same seed and
    template order, so position i in each list corresponds structurally."""
    n = min(len(en_prompts), len(fr_prompts))
    return list(zip(en_prompts[:n], fr_prompts[:n]))


def capture_payload_at_heads(mdl, tok, donor_prompts, heads):
    payloads = []
    for p in donor_prompts:
        enc = tok(p["text"], return_tensors="pt").to(DEVICE)
        store = {}
        with head_hooks(mdl, cfg_for_model(mdl), mode="capture",
                        heads=heads, capture_store=store):
            _ = mdl(**enc)
        # store: {(L, H): tensor [1, head_dim]} → flatten to [head_dim]
        payloads.append({k: v.squeeze(0) for k, v in store.items()})
    return payloads


def logit_diffs_with_patch(mdl, tok, target_prompts, per_prompt_payload):
    assert len(target_prompts) == len(per_prompt_payload)
    diffs = np.zeros(len(target_prompts), dtype=np.float32)
    for i, (p, payload) in enumerate(zip(target_prompts, per_prompt_payload)):
        enc = tok(p["text"], return_tensors="pt").to(DEVICE)
        with head_hooks(mdl, cfg_for_model(mdl), mode="patch",
                        heads=list(payload.keys()), patch_payload=payload):
            out = mdl(**enc)
        logits = out.logits[0, -1]
        c_tok = tok.encode(p["correct"],   add_special_tokens=False)[0]
        i_tok = tok.encode(p["incorrect"], add_special_tokens=False)[0]
        diffs[i] = float(logits[c_tok] - logits[i_tok])
    return diffs


def patching_study(mk, mp, topk=TOPK_HEADS, seed=0):
    mdl, tok, cfg = load_model(mp)
    en_verbs = verify_verb_pairs(tok, EN_VERB_PAIRS)
    fr_verbs = verify_verb_pairs(tok, FR_VERB_PAIRS)
    cond = build_all_conditions(seed=seed)
    en_prompts = filter_prompts_by_verbs(cond["en_nat"],     en_verbs)
    fr_prompts = filter_prompts_by_verbs(cond["fr_matched"], fr_verbs)

    # second EN prompt set with a different seed → same-language different-template floor
    cond2 = build_all_conditions(seed=seed + 100)
    en_prompts_alt = filter_prompts_by_verbs(cond2["en_nat"], en_verbs)

    en_rank = rank_or_load(mk, "en_nat", mdl, tok, en_prompts, cfg)
    en_heads = [(L, H) for (L, H, _s) in en_rank["ranked"][:topk]]

    pairs_cross = pair_prompts(en_prompts, fr_prompts)
    pairs_floor = pair_prompts(en_prompts, en_prompts_alt)

    def do_pair(pairs, label):
        donors  = [a for (a, _b) in pairs]
        targets = [b for (_a, b) in pairs]
        clean   = logit_diffs(mdl, tok, targets).mean()
        ablated = logit_diffs_with_hooks(mdl, tok, targets, mode="ablate", heads=en_heads).mean()
        payload = capture_payload_at_heads(mdl, tok, donors, en_heads)
        patched = logit_diffs_with_patch(mdl, tok, targets, payload).mean()
        denom = clean - ablated
        compat = float((patched - ablated) / denom) if abs(denom) > 1e-6 else float("nan")
        print(f"  [{label}] clean={clean:+.3f} ablated={ablated:+.3f} patched={patched:+.3f} compat={compat:+.3f}")
        return {"clean": float(clean), "ablated": float(ablated),
                "patched": float(patched), "compat": compat, "n": len(pairs)}

    print(f"[{mk}] agreement patching:")
    cross = do_pair(pairs_cross, "EN→FR_matched (agreement)")
    floor = do_pair(pairs_floor, "EN→EN  diff-template (floor)")

    out = {"topk": topk, "en_heads": en_heads, "cross": cross, "floor": floor}
    with open(OUT_DIR / f"{mk}_patching.json", "w") as f:
        json.dump(out, f, indent=2)

    unload(mdl)
    return out


# To run on all four scales:
# for mk, mp in MODEL_PATHS.items():
#     patching_study(mk, mp)


## Cell 11 — Section E — IOI vs Agreement side-by-side

Loads `exp14_results/<model>.json` for the IOI compat numbers and this
notebook's `exp15_results/<model>_patching.json` for agreement compat. Produces
the (IOI − agreement) gap per scale — the headline the ε framework stands or
falls on.

If IOI compat ≈ 1.0 at all scales (per Exp14) and agreement compat drops with
scale, ε is confirmed as task-dependent. If agreement compat also sits ~1.0, the
framework is in trouble: there is no scale-dependent task-splitting at the
computation level, and the dissociation Lim et al. observe would have to live
somewhere else (representation-only, which Exp13's subspace arm couldn't
measure).


In [12]:
def load_ioi_compat(mk):
    f = EXP14_DIR / f"{mk}.json"
    if not f.exists():
        return None
    try:
        with open(f) as fh:
            d = json.load(fh)
        # The exact field name depends on how Exp14 saved it.
        # Common shapes: d["compat"]["EN→FR"] or d["cross"]["compat"].
        # Try a few:
        for path in [("compat", "EN→FR"), ("compat", "en_to_fr"),
                     ("cross", "compat"), ("en_to_fr", "compat")]:
            node = d
            try:
                for k in path:
                    node = node[k]
                return float(node)
            except (KeyError, TypeError):
                continue
        print(f"  [{mk}] exp14 result found but unknown field layout — inspect manually.")
        return None
    except Exception as e:
        print(f"  [{mk}] couldn't read exp14 result: {e}")
        return None


def compile_comparison():
    rows = []
    for mk in MODEL_PATHS:
        agr_f = OUT_DIR / f"{mk}_patching.json"
        if not agr_f.exists():
            print(f"  [{mk}] no agreement patching result yet, skipping.")
            continue
        with open(agr_f) as f:
            agr = json.load(f)
        ioi = load_ioi_compat(mk)
        row = {
            "model": mk,
            "agr_compat_cross": agr["cross"]["compat"],
            "agr_compat_floor": agr["floor"]["compat"],
            "ioi_compat_cross": ioi,
            "gap_ioi_minus_agr": (ioi - agr["cross"]["compat"]) if ioi is not None else None,
        }
        rows.append(row)

    print(f"{'model':10s} {'IOI compat':>12s} {'agr compat':>12s} {'floor':>8s} {'gap':>8s}")
    for r in rows:
        ioi_s = f"{r['ioi_compat_cross']:+.3f}" if r['ioi_compat_cross'] is not None else "  n/a"
        gap_s = f"{r['gap_ioi_minus_agr']:+.3f}" if r['gap_ioi_minus_agr'] is not None else "  n/a"
        print(f"{r['model']:10s} {ioi_s:>12s} {r['agr_compat_cross']:+.3f}     {r['floor_compat'] if 'floor_compat' in r else r['agr_compat_floor']:+.3f}  {gap_s:>8s}")

    with open(OUT_DIR / "comparison.json", "w") as f:
        json.dump(rows, f, indent=2)
    return rows


# Run after patching_study has produced files for every scale:
# compile_comparison()


## Cell 12 — Run order

Call order to go from zero to the final table:

1. Cell 6 (baselines). Preflight. If `fr_matched` acc < 0.75 at some scale, the
   cross-language test won't be meaningful there — adjust nouns/templates
   before continuing. Natural EN/FR expected to work.
2. Cell 8 (head ranking) — expensive, cached. Uncomment and run the loop.
3. Cell 9 (ablation study) — Exp13-style table for agreement. Loop over scales.
4. Cell 10 (patching study) — Exp14-style compat, with floor. Loop over scales.
5. Cell 11 (comparison) — IOI vs agreement gap vs scale.

**Decision rule.** If floor control compat ≠ ~1.0 at some scale, do not read
cross-language numbers at that scale — the method isn't passing its own sanity.
Fix (template matching, pairing index, hook placement) and retry before
interpretation. This is the Exp14 pattern that fixed Exp10–12.

**Known risks worth watching for in the output.**
- Single-token verb loss on some scales (tokenizer drops some pairs) → fewer
  prompts per cell, higher variance. Mitigated by Cell 4's filter.
- FR-matched baseline acc dropping too low in base models (zero-shot code-mixed
  prompts) → may need to switch to natural FR for base models and only use
  matched for -it models. First-pass check in Cell 6.
- 27B agreement baseline may still be high enough that ε-ablation hurts
  everything (saturated circuit). If so, raise topk to compensate.
